In [1]:
! pip install pytorch_lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 71.9 MB/s eta 0:00:00


Подключаем все библиотеки

In [2]:
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torchmetrics.classification import MulticlassF1Score, MulticlassAUROC


SEED = 2025
pl.seed_everything(SEED, workers=True)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


INFO:lightning_fabric.utilities.seed:Seed set to 2025


загрузка датасета, предобработка (ToTensor + Normalize), разбиение на train / val / test, оздание DataLoader'ов



 Простая сверточная нейросеть:
 1) 2 сверточных слоя для извлечения признаков,
 2) MaxPooling уменьшает размерность,
 3) Полносвязные слои выполняют классификацию

In [3]:
class FashionMNISTDataModule(pl.LightningDataModule):
    def __init__(self, batch_size=64, data_dir="./data"):
        super().__init__()
        self.batch_size = batch_size
        self.data_dir = data_dir

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def prepare_data(self):
        datasets.FashionMNIST(self.data_dir, train=True, download=True)
        datasets.FashionMNIST(self.data_dir, train=False, download=True)

    def setup(self, stage=None):
        full_train = datasets.FashionMNIST(
            self.data_dir, train=True, transform=self.transform
        )
        self.test_dataset = datasets.FashionMNIST(
            self.data_dir, train=False, transform=self.transform
        )

        train_size = int(0.9 * len(full_train))
        val_size = len(full_train) - train_size

        self.train_dataset, self.val_dataset = random_split(
            full_train,
            [train_size, val_size],
            generator=torch.Generator().manual_seed(SEED)
        )

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size)


Класс модели на PyTorch Lightning, описываем архитектуру нейросети, задаём функцию потерь

In [4]:
class FashionMNISTModel(pl.LightningModule):
    def __init__(self, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

        self.criterion = nn.CrossEntropyLoss()

        self.val_f1 = MulticlassF1Score(num_classes=10, average="macro")
        self.val_auc = MulticlassAUROC(num_classes=10)

        self.test_f1 = MulticlassF1Score(num_classes=10, average="macro")
        self.test_auc = MulticlassAUROC(num_classes=10)

    def forward(self, x):
        return self.net(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        probs = torch.softmax(logits, dim=1)

        self.val_f1(probs, y)
        self.val_auc(probs, y)

        self.log("val_loss", loss, prog_bar=True)
        self.log("val_f1", self.val_f1, prog_bar=True)
        self.log("val_auc", self.val_auc, prog_bar=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        probs = torch.softmax(logits, dim=1)

        self.test_f1(probs, y)
        self.test_auc(probs, y)

        self.log("test_f1", self.test_f1)
        self.log("test_auc", self.test_auc)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)  # Используем Adam — устойчивый и популярный оптимизатор
        return optimizer                                                     # learning rate выбран эмпирически


Создаём Trainer: EarlyStopping останавливает обучение, если val_loss не улучшается, TensorBoardLogger сохраняет логи, обучение идёт на GPU, если он доступен

In [5]:
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

datamodule = FashionMNISTDataModule(batch_size=64)
model = FashionMNISTModel(lr=1e-3)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    mode="min"
)

logger = TensorBoardLogger("tb_logs", name="fashion_mnist")

trainer = pl.Trainer(
    max_epochs=10,
    callbacks=[early_stopping],
    logger=logger,
    accelerator="auto"
)

trainer.fit(model, datamodule=datamodule)


INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
100%|██████████| 26.4M/26.4M [00:02<00:00, 10.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 177kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.31MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.0MB/s]
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ net       │ Sequential        │  421 K │ train │     0 │
│ 1 │ criterion │ CrossEntropyLoss  │      0 │ train │     0 │
│ 2 │ val_f1    │ MulticlassF1Score │      0 │ train │     0 │
│ 3 │ val_auc   │ MulticlassAUROC   │      0 │ train │     0 │
│ 4 │ test_f1   │ MulticlassF1Score │      0 │ train │     0 │
│ 5 │ test_auc  │ MulticlassAUROC   │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 421 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 421 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: No positive samples in 
targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=10` reached.


Проверяем качество модели на тестовой выборке

In [6]:
trainer.test(model, datamodule=datamodule)


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_auc          │    0.9945632815361023     │
│          test_f1          │    0.9168654084205627     │
└───────────────────────────┴───────────────────────────┘

[{'test_f1': 0.9168654084205627, 'test_auc': 0.9945632815361023}]

Результаты получились хорошими, успех😎